# Multi-Region Cell Segmentation Pipeline
# Sequential Processing: Scripts 6a → 6b → 6c

## Overview
This pipeline performs advanced multi-region segmentation of FRET cells by identifying different cellular compartments based on brightness patterns and vesicle detection. The analysis separates cells into multiple functional regions to study heterogeneous protein distributions and dynamics.

## Biological Background
Membrone proteins often show:
- **Brightness heterogeneity**: Different molecular aggregation states
- **Mobile vesicles**: Fast-moving organelles in red-delay channel  
- **Compartmentalization**: Distinct regions with different protein concentrations

## Pipeline Structure
1. **Script 6a**: Export red-delay time series for vesicle detection
2. **Script 6b**: Combine probability maps from external analysis (Ilastik)
3. **Script 6c**: Generate multiple segmentation masks using two approaches

## Input/Output Summary
- **Input**: `*.ptu` files, N&B analysis results, Ilastik probability maps
- **Output**: Multiple region masks (`*_vesicles.tif`, `*_low.tif`, `*_high.tif`, `*_lowNB.tif`, `*_highNB.tif`)

## External Dependencies
- **Ilastik software**: For vesicle detection using "cell counting" approach
- **tttrlib**: For TTTR data processing

---

## SCRIPT 6C: MULTI-REGION SEGMENTATION

## Multi-Region Cell Segmentation Pipeline

### Overview
Generates multiple complementary segmentation masks for each cell using two different approaches:
1. **Intensity-based segmentation**: Separates vesicles, low-intensity, and high-intensity regions
2. **Number & Brightness segmentation**: Separates regions based on molecular brightness values

### Segmentation Approaches

#### 1. Intensity-Based Approach
- **Vesicle mask**: Bright, mobile structures detected by Ilastik (threshold ≥ 1)
- **Low/High intensity**: Otsu thresholding on green channel sum (excluding vesicles)
- **Biological meaning**: Separates membrane regions by protein concentration

#### 2. Number & Brightness Approach  
- **Low/High N&B**: Fixed brightness threshold at 1.375
- **Biological meaning**: Separates monomeric (B<1.375) vs aggregated (B≥1.375) proteins

### Input/Output
- **Input**: 
  - `*_ROI.tif` (cell masks from initial segmentation)
  - `*_red_delay_Probabilities.tif` (vesicle probability maps, for simplicity move these files into the measurement folder)
  - `*_green_p.tif`, `*_green_s.tif` (green channel time series)
  - `*_brightness.tif` (from N&B analysis)
- **Output**:
  - `*_vesicles.tif` (vesicle mask)
  - `*_low.tif`, `*_high.tif` (intensity-based masks)
  - `*_lowNB.tif`, `*_highNB.tif` (brightness-based masks)

### Configuration Parameters
- **Vesicle threshold**: ≥ 1 (probability from Ilastik)
- **N&B threshold**: 1.375 (brightness cutoff)
- **Intensity threshold**: Otsu automatic (data-driven)

## Load required libraries

In [10]:
import numpy as np
from skimage.filters import threshold_otsu
from skimage import io
import glob
import os

## Load data

In [11]:
# Define input file pattern
# Process only ROI1 files (primary cell masks)  
path = 'C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Cells/DA*/*_ROI1.tif'

## Process folder

In [12]:
# Initialize storage lists
list_filenames = list()  # filenames
temp = list()
list_thresholds = list()

for file in glob.glob(path):
    # Extract base filename (remove ROI suffix)
    filename = os.path.abspath(file).split("_ROI")[0]
    
    # Load input data
    cell_mask = io.imread(file).astype(np.uint8)          # Single-frame cell boundary
    vesicles = io.imread(filename + '_red_delay_Probabilities.tif') # Time series (50 frames)
    green_p = io.imread(filename + '_green_p.tif')        # Green p-polarized time series
    green_s = io.imread(filename + '_green_s.tif')        # Green s-polarized time series
    
    # Extract output filename base
    save_as = os.path.abspath(file).split(".")[0]
    
    # =========================================================================
    # INTENSITY-BASED SEGMENTATION
    # =========================================================================
    
    # Extend 2D cell mask to match time series dimensions
    extended_cell_mask = np.tile(cell_mask, (vesicles.shape[0], 1, 1))

    # Generate vesicle mask: pixels inside cells with high vesicle probability
    vesicle_mask = np.where((extended_cell_mask > 0) & (vesicles >= 1), 1, 0)
    
    # Save as vesicle mask
    io.imsave(save_as + '_vesicle.tif', vesicle_mask.astype(np.uint8), check_contrast=False)
    
    # Generate cleaned ROI: remove vesicle pixels from cell mask
    cleaned_roi = np.where(vesicle_mask == 1, 0, extended_cell_mask)
    
    # Calculate green channel sum for intensity-based segmentation
    green_total = green_p + green_s
    ROI_image = green_total * cleaned_roi
 
    # Sum intensities across all time frames for stable thresholding
    ROI_sum = np.array(np.sum(ROI_image, axis=0), dtype=np.uint16)
    
    # Build histogram excluding background pixels (value = 0)
    ROI_bin_values = np.bincount(ROI_sum.flatten())
    ROI_bin_values_z = np.copy(ROI_bin_values)
    ROI_bin_values_z[0] = 0   # Remove background from histogram
    
    # Apply Otsu thresholding to separate low/high intensity regions
    thresh = threshold_otsu(ROI_sum, hist=ROI_bin_values_z)
    
    # Generate low & high mask - extend for the required number of frames
    ROI_sum_extended = np.tile(ROI_sum, (vesicles.shape[0], 1, 1))
    
    # Generate the low & high intensity masks
    mask_low_intensity = np.where((ROI_sum < thresh) & (cleaned_roi > 0), 1, 0)
    mask_high_intensity = np.where((ROI_sum >= thresh) & (cleaned_roi > 0), 1, 0)

    # Save masks
    io.imsave(save_as + '_low.tif', mask_low_intensity.astype(np.uint8), check_contrast=False)
    io.imsave(save_as + '_high.tif', mask_high_intensity.astype(np.uint8), check_contrast=False)
     
    # =========================================================================
    # NUMBER & BRIGHTNESS SEGMENTATION
    # =========================================================================
    
    # Load brightness image from previous N&B analysis
    img_brightness = io.imread( filename + '_brightness.tif')
    
    # Fixed threshold for molecular brightness
    # 1.375 separates monomeric (low) from aggregated (high) species
    # Threshold based on manual inspection of the brightness histogram
    NB_threshold = 1.375
    
    # Generate N&B-based masks (single frame, no time dimension)
    mask_low_NB= np.where((img_brightness < NB_threshold) & (cell_mask > 0), 1, 0)
    mask_high_NB= np.where((img_brightness >= NB_threshold) & (cell_mask > 0), 1, 0)
    
    # Save N&B-based masks
    io.imsave(save_as + '_lowNB.tif', mask_low_NB.astype(np.uint8), check_contrast=False)
    io.imsave(save_as + '_highNB.tif', mask_high_NB.astype(np.uint8), check_contrast=False)
    
    print('Processing....' + filename)
    


Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_64
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_66
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_68
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_70
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_72
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_74
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_76


## SUMMARY OF OUTPUT MASKS


Generated mask files for each cell:

INTENSITY-BASED APPROACH:
- *_vesicles.tif    : Vesicle/organelle mask (mobile structures)
- *_low.tif    : Low-intensity cytoplasmic regions  
- *_high.tif   : High-intensity cytoplasmic regions

NUMBER & BRIGHTNESS APPROACH:
- *_lowNB.tif  : Low brightness regions (B < 1.375, likely monomers)
- *_highNB.tif : High brightness regions (B ≥ 1.375, likely aggregates)

These masks can be used independently or in combination for:
- Quantitative analysis of protein distributions
- FRET efficiency calculations in different compartments  
- Molecular dynamics studies
- Aggregation state analysis